# mp-spawn-workers — faded example 3: Pass extra arguments through mp.spawn args tuple to each worker

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `mp-spawn-workers`. Running the beacon reports progress on the `Distributed: mp.spawn workers` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: mp.spawn workers` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`mp-spawn-workers`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "mp-spawn-workers"
DD_SUBTOPIC = "Distributed: mp.spawn workers"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Extra data needed by each worker process is passed through `mp.spawn`'s `args` keyword argument as a tuple. Each worker receives these as positional arguments after `rank`. Using a shared `Manager().list()` for result collection is a clean pattern that avoids file I/O and works across processes since the Manager provides a server-backed shared object.

## Faded exercise 3

Implement `spawn_with_config(spawn_module, worker_fn, nprocs, config_dict, result_list)` that calls `spawn_module.spawn` with:
- `fn = worker_fn`
- `args = (nprocs, config_dict, result_list)` (passing both config and result collector)
- `nprocs = nprocs`
- `join = True`

Your task: **fill in the spawn call with the correct args tuple and join=True**.

**Fill in:** The spawn call passing args=(nprocs, config_dict, result_list), nprocs=nprocs, join=True.

In [ ]:
def spawn_with_config(spawn_module, worker_fn, nprocs: int, config_dict: dict, result_list):
    spawn_module.spawn(worker_fn, args=(nprocs, config_dict, result_list), nprocs=nprocs, join=True)

def _test():
    class _FakeSpawn:
        def __init__(self): self.last = {}
        def spawn(self, fn, args, nprocs, join):
            self.last = dict(fn=fn, args=args, nprocs=nprocs, join=join)

    fake = _FakeSpawn()
    fn = lambda rank, ws, cfg, res: None
    cfg = {'lr': 0.01}
    res = []
    spawn_with_config(fake, fn, nprocs=4, config_dict=cfg, result_list=res)
    assert fake.last['nprocs'] == 4
    assert fake.last['join'] == True
    assert fake.last['args'] == (4, cfg, res)


def _test():
    class _FakeSpawn:
        def __init__(self): self.calls = []
        def spawn(self, fn, args, nprocs, join):
            self.calls.append(dict(fn=fn, args=args, nprocs=nprocs, join=join))

    fake = _FakeSpawn()
    fn = lambda rank, ws, cfg, res: None
    cfg = {'lr': 0.001, 'batch': 32}
    res = []
    spawn_with_config(fake, fn, nprocs=2, config_dict=cfg, result_list=res)
    assert len(fake.calls) == 1
    c = fake.calls[0]
    assert c['nprocs'] == 2, f"nprocs should be 2, got {c['nprocs']}"
    assert c['join'] == True, "join should be True"
    assert c['args'][0] == 2, "first arg should be world_size=nprocs=2"
    assert c['args'][1] is cfg, "second arg should be config_dict"
    assert c['args'][2] is res, "third arg should be result_list"
    assert c['fn'] is fn


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def spawn_with_config(spawn_module, worker_fn, nprocs: int, config_dict: dict, result_list):
    spawn_module.spawn(worker_fn, args=(nprocs, config_dict, result_list), nprocs=nprocs, join=True)

def _test():
    class _FakeSpawn:
        def __init__(self): self.last = {}
        def spawn(self, fn, args, nprocs, join):
            self.last = dict(fn=fn, args=args, nprocs=nprocs, join=join)

    fake = _FakeSpawn()
    fn = lambda rank, ws, cfg, res: None
    cfg = {'lr': 0.01}
    res = []
    spawn_with_config(fake, fn, nprocs=4, config_dict=cfg, result_list=res)
    assert fake.last['nprocs'] == 4
    assert fake.last['join'] == True
    assert fake.last['args'] == (4, cfg, res)
```
</details>